# Balanced threshold run: Real-Time Flood Data Pipeline

The first tuning run (`final_tuning.ipynb`) chose a good network (best test AP so far, 0.567) but its threshold,
set for best day-level F1 on three small development folds (20 flood events), was very cautious: precision 0.88 and
recall 0.30 on 2024-2025. A warning system needs precision and recall in balance, so this run changes how the
threshold and the number of epochs are chosen. The network setting is kept from the first run.

Run with **Runtime > Run all** on a T4 GPU, in a fresh session.

**Plan, fixed before any result is seen**

1. Network (from the first run, unchanged): LSTM with 64 units, dropout 0.5, a 14-day window, learning rate 0.001,
   batch size 64 and event weights.
2. Choosing data: leave-one-year-out across the labelled years 2015, 2016, 2018, 2019, 2021, 2022 and 2023 (2017 and
   2020 have no usable labels). Each year is scored by models trained on the other six, giving out-of-fold scores for
   all 34 events and 109 flood days before the test period. Eight seeds per fold, scores averaged.
3. Epochs (to limit overfitting): pooled out-of-fold average precision is measured after every epoch from 1 to 15
   and smoothed with a centred three-epoch moving average. The epoch with the highest smoothed value is chosen;
   ties (within 0.001) go to fewer epochs.
4. Threshold (balance): at the chosen epoch, the threshold where out-of-fold day-level precision and recall are
   closest (the break-even point, where false alarm days equal missed flood days), among development-score quantiles
   0.80 to 0.999 that catch at least one flood day. Ties go to the higher F1.
5. Provisional model (no CHIRPS): the same network, epochs and threshold rules on its own out-of-fold scores.
6. The final test cell is run **once** and its result is reported whatever it is, together with the earlier results
   (v2, the 2-epoch model and v3).

In [ ]:
# 1. Setup
import os, json, time, zipfile
os.environ["TF_DETERMINISTIC_OPS"] = "1"
import numpy as np, pandas as pd, tensorflow as tf
tf.get_logger().setLevel("ERROR")
from numpy.lib.stride_tricks import sliding_window_view
print("TensorFlow", tf.__version__, "| Keras", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "")
print("GPU:", tf.config.list_physical_devices("GPU") or "none (Runtime > Change runtime type > T4 GPU)")

In [ ]:
# 2. Data: upload features_daily_audited.csv from
#    data_collection/data/08_model_ready/ in the project folder, or point DATA_PATH at a copy in Google Drive.
from google.colab import drive, files
DATA_PATH = "/content/drive/MyDrive/flood_tuning/features_daily_audited.csv"   # optional Drive copy
OUT_DIR = "/content/drive/MyDrive/flood_tuning/balanced_results"                       # results saved here if Drive is used
try:
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted:", e)
if not os.path.exists(DATA_PATH):
    print("Upload features_daily_audited.csv")
    up = files.upload()
    DATA_PATH = "/content/" + next(iter(up))
    OUT_DIR = "/content/balanced_results"
os.makedirs(OUT_DIR, exist_ok=True)
RAW = pd.read_csv(DATA_PATH, parse_dates=["date"]).sort_values(["catchment", "date"])
print(len(RAW), "rows,", RAW["date"].min().date(), "to", RAW["date"].max().date())
print(RAW["label_status"].value_counts().to_dict())

In [ ]:
# 3. Shared code, copied from the repository (sequences.py, save_final_model.py, event_train_evaluate.py)
SEQ_FEATS = ["rain_mm", "imerg_max_rate", "imerg_total_mm", "imerg_hours_ge5", "era5_rain_mm", "soil_1", "soil_2",
             "temp_c", "dewpoint_depression", "pressure_hpa", "month_sin", "month_cos"]
CATCHMENTS = ["ngong", "mathare", "nairobi"]
DEV_FOLDS = [(range(2015, 2019), [2019]), (range(2015, 2021), [2021, 2022]), (range(2015, 2023), [2023])]
FINAL_FOLD = (range(2015, 2024), [2024, 2025])
SEEDS = list(range(8))
DAY = pd.Timedelta(days=1)
QS = np.round(np.arange(0.90, 0.9991, 0.001), 3)
_windows = {}


def make_windows(t, drop=()):
    # one window per catchment-day: the t days ending on that day, labelled with the last day
    if (t, drop) not in _windows:
        df = RAW.copy()
        for c in CATCHMENTS:
            df[f"is_{c}"] = (df["catchment"] == c).astype("float32")
        feats = [f for f in SEQ_FEATS + [f"is_{c}" for c in CATCHMENTS] if f not in drop]
        xs, metas = [], []
        for c, g in df.groupby("catchment"):
            vals = g[feats].values.astype("float32")
            xs.append(sliding_window_view(vals, (t, vals.shape[1]))[:, 0])
            metas.append(g.iloc[t - 1:][["catchment", "date", "label_status", "event_id"]].copy())
        meta = pd.concat(metas).reset_index(drop=True)
        meta["year"] = meta["date"].dt.year
        _windows[(t, drop)] = (np.concatenate(xs), meta, feats)
    return _windows[(t, drop)]


def make_model(n_feats, t, units, dropout, lr):
    m = tf.keras.Sequential([
        tf.keras.layers.Input((t, n_feats)),
        tf.keras.layers.LSTM(units),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="binary_crossentropy")
    return m


def fold_data(X, meta, train_years, val_years):
    st = meta["label_status"].values
    flood, known = st == "flood", np.isin(st, ["flood", "no_flood"])
    tr = meta["year"].isin(list(train_years)).values & known
    va = meta["year"].isin(list(val_years)).values
    flat = X[tr].reshape(-1, X.shape[2])
    mu, sd = flat.mean(0), flat.std(0) + 1e-6
    w = float(np.sqrt((tr & ~flood).sum() / (tr & flood).sum()))          # class weight
    eid = meta["event_id"].where(flood)
    L = meta[tr & flood].groupby(eid[tr & flood]).size()                   # days per event
    sw = np.where(flood, w * L.mean() / eid.map(L).fillna(1).values, 1.0)  # event weights
    return (X[tr] - mu) / sd, flood[tr].astype("float32"), sw[tr], (X[va] - mu) / sd, va, mu, sd


def run_setting(t, units, dropout, lr, max_ep, folds, drop=(), keep_models=False, log=None):
    # eight seeds; scores after every epoch, averaged over seeds -> {epoch: DataFrame(val, catchment, date, score)}
    X, meta, feats = make_windows(t, drop)
    out, models = {}, []
    for train_years, val_years in folds:
        Xtr, ytr, sw, Xva, va, mu, sd = fold_data(X, meta, train_years, val_years)
        S = np.zeros((max_ep, len(Xva)))
        for seed in SEEDS:
            tf.keras.utils.set_random_seed(seed)
            m = make_model(X.shape[2], t, units, dropout, lr)
            for ep in range(max_ep):
                h = m.fit(Xtr, ytr, sample_weight=sw, epochs=1, batch_size=64, verbose=0)
                S[ep] += m.predict(Xva, batch_size=1024, verbose=0).ravel() / len(SEEDS)
                if log is not None:
                    log.append({"val": "-".join(map(str, val_years)), "seed": seed, "epoch": ep + 1,
                                "loss": h.history["loss"][0]})
            if keep_models:
                models.append(m)
        base = meta.loc[va, ["catchment", "date"]].reset_index(drop=True)
        for ep in range(max_ep):
            d = base.copy(); d["val"] = "-".join(map(str, val_years)); d["score"] = S[ep]
            out.setdefault(ep + 1, []).append(d)
    out = {ep: pd.concat(v, ignore_index=True) for ep, v in out.items()}
    return (out, models, feats, mu, sd) if keep_models else out


LAB = RAW[["catchment", "date", "label_status", "event_id"]]


def prepare(d):
    m = d.merge(LAB, on=["catchment", "date"])
    parts = []
    for _, g in m.groupby("val"):
        ev = (g[g["label_status"] == "flood"].groupby("event_id")
              .agg(catchment=("catchment", "first"), start=("date", "min"), end=("date", "max"), days=("date", "size")))
        known = g[g["label_status"].isin(["flood", "no_flood"])]
        parts.append((g, ev, known["label_status"].eq("flood").values, known["score"].values))
    return parts


def ap(y, s):
    o = np.argsort(-s, kind="mergesort")
    y, s = y[o], s[o]
    last = np.r_[np.diff(s) != 0, True]
    tp = np.cumsum(y)[last]
    return float(np.sum(np.diff(np.r_[0, tp / tp[-1]]) * tp / np.arange(1, len(y) + 1)[last]))


def pooled_ap(parts):
    return ap(np.concatenate([p[2] for p in parts]), np.concatenate([p[3] for p in parts]))


def stats(parts, thr, cy):
    n = det = det1 = n1 = eps = false_eps = tp = fp = fn = 0
    for g, ev, y, s in parts:
        a = s >= thr
        tp += int((a & y).sum()); fp += int((a & ~y).sum()); fn += int((~a & y).sum())
        alarms = g[g["score"] >= thr]
        for e in ev.itertuples():
            x = alarms[alarms["catchment"] == e.catchment]
            hit = bool(((x["date"] >= e.start - DAY) & (x["date"] <= e.end + DAY)).any())
            det += hit
            if e.days == 1:
                n1 += 1; det1 += hit
        n += len(ev)
        for c, x in alarms.sort_values("date").groupby("catchment"):
            evc = ev[ev["catchment"] == c]
            for _, ep in x.groupby((x["date"].diff() != DAY).cumsum()):
                eps += 1
                false_eps += not ((evc["start"] - DAY <= ep["date"].max()) & (evc["end"] + DAY >= ep["date"].min())).any()
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn)
    er = det / n
    ep_p = (eps - false_eps) / eps if eps else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": p, "recall": r, "f1": 2 * p * r / (p + r) if p + r else 0.0,
            "events": n, "detected": det, "one_day_detected": f"{det1}/{n1}", "multi_day_detected": f"{det - det1}/{n - n1}",
            "event_recall": er, "episodes": eps, "false_episodes": false_eps, "episode_precision": ep_p,
            "event_f1": 2 * ep_p * er / (ep_p + er) if ep_p + er else 0.0, "false_eps_per_cy": false_eps / cy}


def pick_threshold(dev_scores):
    dp = prepare(dev_scores)
    grid = np.unique(np.quantile(dev_scores["score"], QS))
    sw = pd.DataFrame([{"threshold": t, **stats(dp, t, 12)} for t in grid])
    return sw.sort_values(["f1", "threshold"], ascending=[False, False]).iloc[0]


def wilson(k, n, z=1.96):
    p = k / n; c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return round(c - h, 2), round(c + h, 2)


def loss_history(t, units, dropout, lr, max_ep, fold, seeds, drop=()):
    # training and validation loss per epoch on one fold, averaged over seeds; both weighted the same way
    X, meta, _ = make_windows(t, drop)
    Xtr, ytr, sw, Xva, va, _, _ = fold_data(X, meta, *fold)
    st = meta["label_status"].values
    flood = st == "flood"
    tr_years = meta["year"].isin(list(fold[0])).values & np.isin(st, ["flood", "no_flood"])
    w = float(np.sqrt((tr_years & ~flood).sum() / (tr_years & flood).sum()))
    eid = meta["event_id"].where(flood)
    L = meta[va & flood].groupby(eid[va & flood]).size()
    known = np.isin(st[va], ["flood", "no_flood"])
    yva = flood[va].astype("float32")
    swva = np.where(yva == 1, w * L.mean() / eid[va].map(L).fillna(1).values, 1.0)
    hist = []
    for seed in seeds:
        tf.keras.utils.set_random_seed(seed)
        m = make_model(X.shape[2], t, units, dropout, lr)
        h = m.fit(Xtr, ytr, sample_weight=sw, epochs=max_ep, batch_size=64, verbose=0,
                  validation_data=(Xva[known], yva[known], swva[known]))
        hist.append([h.history["loss"], h.history["val_loss"]])
    return np.array(hist).mean(0)

print("ready")

In [ ]:
# 4. Leave-one-year-out scores after every epoch (about 25 minutes per model on a T4)
LABELLED_YEARS = [2015, 2016, 2018, 2019, 2021, 2022, 2023]
LOYO = [([y2 for y2 in LABELLED_YEARS if y2 != y], [y]) for y in LABELLED_YEARS]
NET = {"lr": 0.001, "window": 14, "units": 64, "dropout": 0.5}
MAX_EP = 15
CY_DEV = 3 * len(LABELLED_YEARS)


def choose(kind, drop=()):
    f = f"{OUT_DIR}/loyo_scores_{kind}.csv"
    if os.path.exists(f):
        allsc = pd.read_csv(f, parse_dates=["date"])
        scores = {ep: g.drop(columns="epoch") for ep, g in allsc.groupby("epoch")}
    else:
        t0 = time.time()
        scores = run_setting(NET["window"], NET["units"], NET["dropout"], NET["lr"], MAX_EP, LOYO, drop=drop)
        pd.concat([s.assign(epoch=ep) for ep, s in scores.items()]).to_csv(f, index=False)
        print(f"{kind}: scored in {time.time() - t0:.0f}s")
    curve = pd.DataFrame([{"epoch": ep, "oof_ap": pooled_ap(prepare(s))} for ep, s in sorted(scores.items())])
    curve["smoothed"] = curve["oof_ap"].rolling(3, center=True, min_periods=2).mean()
    top = curve["smoothed"].max()
    ep = int(curve[curve["smoothed"] >= top - 0.001]["epoch"].min())
    s = scores[ep]
    parts = prepare(s)
    grid = np.unique(np.quantile(s["score"], np.round(np.arange(0.80, 0.9991, 0.001), 3)))
    sw = pd.DataFrame([{"threshold": t, **stats(parts, t, CY_DEV)} for t in grid])
    sw = sw[sw["tp"] > 0]  # P = R = 0 is not a break-even point
    sw["gap"] = (sw["precision"] - sw["recall"]).abs().round(4)
    b = sw.sort_values(["gap", "f1"], ascending=[True, False]).iloc[0]
    curve.to_csv(f"{OUT_DIR}/loyo_curve_{kind}.csv", index=False)
    print(curve.round(3).to_string(index=False))
    print(f"{kind}: epochs {ep}, threshold {b['threshold']:.4f} | out-of-fold precision {b['precision']:.3f}, "
          f"recall {b['recall']:.3f}, F1 {b['f1']:.3f}, events {int(b['detected'])}/{int(b['events'])}, "
          f"event precision {b['episode_precision']:.3f}")
    return {**NET, "epochs": ep, "threshold": float(b["threshold"]), "oof_precision": float(b["precision"]),
            "oof_recall": float(b["recall"]), "oof_f1": float(b["f1"]), "oof_events": f"{int(b['detected'])}/{int(b['events'])}",
            "oof_ap": float(curve.loc[curve.epoch == ep, "oof_ap"].iloc[0])}


choice = {"full": choose("full"), "fast": choose("fast", ("rain_mm",))}
json.dump(choice, open(f"{OUT_DIR}/balanced_choice.json", "w"), indent=2)
print(json.dumps(choice, indent=1))

In [ ]:
# 5. Charts for the report: out-of-fold AP by epoch, and training and validation loss (2015-2022 -> 2023)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
for kind, colour, label in [("full", "#2a78d6", "Full model"), ("fast", "#eb6834", "Provisional (no CHIRPS)")]:
    c = pd.read_csv(f"{OUT_DIR}/loyo_curve_{kind}.csv")
    ax.plot(c["epoch"], c["oof_ap"], color=colour, lw=1, alpha=0.45)
    ax.plot(c["epoch"], c["smoothed"], color=colour, lw=2, label=f"{label} (3-epoch average)")
    ax.axvline(choice[kind]["epochs"], color=colour, ls="--", lw=1)
ax.set_xlabel("Epoch"); ax.set_ylabel("Out-of-fold average precision, 2015-2023"); ax.legend(frameon=False)
for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/loyo_ap_by_epoch.png"); plt.show()

h = loss_history(NET["window"], NET["units"], NET["dropout"], NET["lr"], MAX_EP, DEV_FOLDS[2], seeds=range(4))
pd.DataFrame({"epoch": range(1, MAX_EP + 1), "train_loss": h[0], "val_loss": h[1]}).to_csv(
    f"{OUT_DIR}/loss_curve.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
ax.plot(range(1, MAX_EP + 1), h[0], color="#2a78d6", lw=2, label="Training loss (2015-2022)")
ax.plot(range(1, MAX_EP + 1), h[1], color="#eb6834", lw=2, label="Validation loss (2023)")
ax.axvline(choice["full"]["epochs"], color="#52514e", ls="--", lw=1)
ax.set_xlabel("Epoch"); ax.set_ylabel("Weighted cross-entropy"); ax.legend(frameon=False)
for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/loss_curve.png"); plt.show()

## Final test: run this cell once

Everything above used 2015-2023 only. This trains on 2015-2023 with the chosen epochs and scores 2024-2025 at the
balanced thresholds. Do not change anything above after running it.

In [ ]:
# 6. FINAL TEST (once), confusion matrices and saving the models in the format the backend loads
ch = json.load(open(f"{OUT_DIR}/balanced_choice.json"))
results = {}
for kind, drop in [("full", ()), ("fast", ("rain_mm",))]:
    c = ch[kind]
    scores, models, feats, mu, sd = run_setting(c["window"], c["units"], c["dropout"], c["lr"], c["epochs"],
                                                [FINAL_FOLD], drop=drop, keep_models=True)
    s = scores[c["epochs"]]
    s.to_csv(f"{OUT_DIR}/final_scores_{kind}.csv", index=False)
    parts = prepare(s)
    r = {"threshold": c["threshold"], "epochs": c["epochs"], "ap": pooled_ap(parts), **stats(parts, c["threshold"], 6)}
    r["tn"] = int(sum(len(p[2]) for p in parts)) - r["tp"] - r["fp"] - r["fn"]
    r["precision_ci"] = wilson(r["tp"], r["tp"] + r["fp"]); r["recall_ci"] = wilson(r["tp"], r["tp"] + r["fn"])
    r["event_recall_ci"] = wilson(r["detected"], r["events"])
    results[kind] = r
    name, version = ("lstm_raw", "v4") if kind == "full" else ("lstm_fast", "v3")
    d = f"/content/models/flood_lstm{'' if kind == 'full' else '_fast'}"
    os.makedirs(d, exist_ok=True)
    for seed, m in zip(SEEDS, models):
        m.save_weights(f"{d}/lstm_raw_seed{seed}.weights.h5")
    json.dump({"name": name, "version": version, "training": "event-weighted", "features": feats,
               "window": c["window"], "units": c["units"], "dropout": c["dropout"], "learning_rate": c["lr"],
               "epochs": c["epochs"], "mean": mu.tolist(), "std": sd.tolist(), "threshold": c["threshold"],
               "threshold_rule": "precision-recall break-even on leave-one-year-out scores, 2015-2023",
               "seeds": SEEDS, "train_years": [2015, 2023], "labels": "features_daily_audited.csv",
               "target": "today", "tuned_in": "colab/balanced_threshold.ipynb"}, open(f"{d}/config.json", "w"), indent=2)
    cm = pd.DataFrame([[r["tp"], r["fn"]], [r["fp"], r["tn"]]], index=["Actual flood", "Actual no flood"],
                      columns=["Predicted flood", "Predicted no flood"])
    cm.to_csv(f"{OUT_DIR}/confusion_matrix_{kind}.csv")
    print(); print(f"{kind} confusion matrix (labelled days, 2024-2025)"); print(cm)
json.dump(results, open(f"{OUT_DIR}/final_results.json", "w"), indent=2, default=float)
print()
print(pd.DataFrame(results).to_string())
with zipfile.ZipFile("/content/balanced_models.zip", "w") as z:
    for root, _, fs in os.walk("/content/models"):
        for f in fs:
            z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), "/content"))
    for f in os.listdir(OUT_DIR):
        if not f.startswith("loyo_scores"):
            z.write(f"{OUT_DIR}/{f}", f"results/{f}")
files.download("/content/balanced_models.zip")